# PySpark reference 7: aggregations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-07-aggregations.ipynb)

`groupBy` and every way to aggregate, how aggregations execute, the full aggregate function catalog, conditional and approximate aggregates, mergeable HLL sketches, pivot and unpivot, rollup, cube and grouping sets, and SQL forms. Practise it in [PySpark practice 7: aggregations](pyspark-practice-07-aggregations.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. It also creates `ads` (ad spend and sales per region) for the statistics entries.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

ads = spark.createDataFrame(
    [("north", 10, 120), ("north", 20, 190), ("north", 30, 310), ("north", 40, 390),
     ("south", 10, 200), ("south", 20, 180), ("south", 30, 220), ("south", None, 210)],
    "region string, spend int, sales int",
)

## Contents

| section | entries |
| --- | --- |
| [Grouping](#grouping) | [`DataFrame.groupBy(*cols)` and `GroupedData`](#dataframegroupbycols-and-groupeddata), [Grouping by expressions, several columns and null keys](#grouping-by-expressions-several-columns-and-null-keys), [Filtering groups (`HAVING`)](#filtering-groups-having), [`DataFrame.agg(*exprs)`: aggregating the whole DataFrame](#dataframeaggexprs-aggregating-the-whole-dataframe), [Keeping groups that have no rows](#keeping-groups-that-have-no-rows), [How Spark executes an aggregation](#how-spark-executes-an-aggregation) |
| [Aggregate functions](#aggregate-functions) | [`count`, `count_distinct`](#count-count_distinct), [`approx_count_distinct`](#approx_count_distinct), [`sum`, `avg` / `mean`, `min`, `max`](#sum-avg--mean-min-max), [`count_if`](#count_if), [`sum_distinct`](#sum_distinct), [`product`](#product), [`try_sum`, `try_avg`](#try_sum-try_avg), [`collect_list` / `array_agg`, `collect_set`](#collect_list--array_agg-collect_set), [`listagg` / `string_agg`](#listagg--string_agg), [`first`, `last`, `any_value`](#first-last-any_value), [`max_by`, `min_by`](#max_by-min_by), [`mode`](#mode), [`median`, `percentile`](#median-percentile), [`percentile_approx` / `approx_percentile`](#percentile_approx--approx_percentile), [`histogram_numeric`](#histogram_numeric), [`stddev`, `variance` and their population versions](#stddev-variance-and-their-population-versions), [`skewness`, `kurtosis`](#skewness-kurtosis), [`corr`, `covar_samp`, `covar_pop`](#corr-covar_samp-covar_pop), [`regr_slope`, `regr_intercept`, `regr_r2`, `regr_count`, `regr_avgx`, `regr_avgy`](#regr_slope-regr_intercept-regr_r2-regr_count-regr_avgx-regr_avgy), [`bool_and` / `every`, `bool_or` / `some`](#bool_and--every-bool_or--some), [`bit_or`, `bit_and`, `bit_xor`](#bit_or-bit_and-bit_xor), [`grouping`, `grouping_id`](#grouping-grouping_id) |
| [Conditional aggregates](#conditional-aggregates) | [`sum(when(...))`, `count_if` and SQL `FILTER`](#sumwhen-count_if-and-sql-filter) |
| [Approximate and mergeable aggregates](#approximate-and-mergeable-aggregates) | [HyperLogLog sketches: `hll_sketch_agg`, `hll_union_agg`, `hll_sketch_estimate`](#hyperloglog-sketches-hll_sketch_agg-hll_union_agg-hll_sketch_estimate) |
| [Pivot and unpivot](#pivot-and-unpivot) | [`GroupedData.pivot(pivot_col, values=None)`](#groupeddatapivotpivot_col-valuesnone), [`DataFrame.unpivot(ids, values, variableColumnName, valueColumnName)` / `melt`](#dataframeunpivotids-values-variablecolumnname-valuecolumnname--melt) |
| [Subtotals: rollup, cube and grouping sets](#subtotals-rollup-cube-and-grouping-sets) | [`rollup`, `cube` and `GROUPING SETS`](#rollup-cube-and-grouping-sets) |
| [Aggregations in SQL](#aggregations-in-sql) | [`GROUP BY`, `GROUP BY ALL`, `HAVING` and `FILTER`](#group-by-group-by-all-having-and-filter) |

## Grouping

### `DataFrame.groupBy(*cols)` and `GroupedData`

Groups rows by one or more columns or expressions and returns a `GroupedData` object. Nothing is computed until you
call one of its aggregation methods, which returns a DataFrame with **one row per distinct combination of the grouping
values**: the grouping columns first, then one column per aggregate. `groupby` is an alias.

| parameter | type | description |
| --- | --- | --- |
| `*cols` | str, Column, or list | grouping keys; any expression is allowed (`F.to_date("ts")`, `F.when(...)`) |

| `GroupedData` method | result |
| --- | --- |
| `.agg(*exprs)` / `.agg(dict)` | the general form: any aggregate expressions, your column names |
| `.count()` | rows per group, in a column named `count` |
| `.sum(*cols)`, `.avg(*cols)` / `.mean`, `.min`, `.max` | shortcuts on numeric columns, named like `sum(amount)` |
| `.pivot(col, values=None)` | turn the values of `col` into columns ([pivot](#groupeddatapivotpivot_col-valuesnone)) |
| `.applyInPandas(func, schema)` | run a pandas function per group ([PySpark reference 12: Python UDFs, pandas and Arrow](pyspark-reference-12-python-udfs-and-pandas.ipynb)) |

**Returns:** `GroupedData`; its methods return a DataFrame.

**Notes**
- All null keys form **one group**.
- The output order is not defined; sort explicitly.
- Every output column must be a grouping key or an aggregate; to carry another column, aggregate it (`first`,
  `max_by`) or group by it too.

In [ ]:
by_status = orders.groupBy("status")
by_status.count().orderBy("status").show()
by_status.agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"), F.round(F.avg("amount"), 2).alias("avg_amount")).orderBy("status").show()
by_status.agg({"amount": "max", "order_id": "count"}).orderBy("status").show()

### Grouping by expressions, several columns and null keys

`groupBy` accepts derived values directly, so there is no need to add a column first. With several keys, one row is
produced per combination that **exists in the data**.

**Notes:** to report combinations that have no rows (every customer × every day), build the combinations with a
`crossJoin` and left join the aggregate onto them.

In [ ]:
orders.groupBy(F.to_date("order_ts").alias("day"), "status").agg(F.count("*").alias("orders")).orderBy("day", "status").show(5)
orders.groupBy(F.when(F.col("amount") >= 200, "large").otherwise("small").alias("band")).count().orderBy("band").show()
orders.join(customers, "customer_id", "left").groupBy("region").agg(F.sum("amount").alias("revenue")).orderBy(F.asc_nulls_last("region")).show()

### Filtering groups (`HAVING`)

A `where` **before** `groupBy` filters input rows (SQL `WHERE`); a `where` **after** `agg` filters groups by their
aggregates (SQL `HAVING`). Filter on the aliases you gave the aggregates.

In [ ]:
(orders.where(F.col("status") != "returned")
 .groupBy("customer_id").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))
 .where((F.col("orders") >= 2) & (F.col("revenue") > 200))
 .orderBy("customer_id").show())

### `DataFrame.agg(*exprs)`: aggregating the whole DataFrame

`df.agg(...)` is `df.groupBy().agg(...)`: one output row for the whole DataFrame. Use it for totals and profiles.

**Notes:** aggregating an empty DataFrame still returns one row: counts are 0, every other aggregate is null.

In [ ]:
orders.agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"), F.min("order_ts").alias("first"), F.max("order_ts").alias("last")).show()
orders.where("amount > 5000").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue")).show()

### Keeping groups that have no rows

`groupBy` cannot invent keys that are not in the data. To list every customer, aggregate the facts first, then **left
join** the result onto the dimension and replace the missing aggregates with explicit zeros.

In [ ]:
per_customer = orders.groupBy("customer_id").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))
(customers.select("customer_id", "name").join(per_customer, "customer_id", "left")
 .select("customer_id", "name", F.coalesce("orders", F.lit(0)).alias("orders"), F.coalesce("revenue", F.lit(0)).alias("revenue"))
 .orderBy("customer_id").show())

### How Spark executes an aggregation

An aggregation runs in two steps: each task computes **partial** aggregates for the keys in its partition
(`partial_sum`, `partial_count`), the partial rows are **shuffled** by key, and a **final** step merges them. Only one
partial row per key per task crosses the network, which is why `groupBy` on a billion rows with few keys is cheap.

| operator | used for |
| --- | --- |
| `HashAggregate` | fixed-size buffers (sum, count, min, max on primitive types) |
| `ObjectHashAggregate` | object buffers (`collect_list`, `collect_set`, `percentile_approx`) |
| `SortAggregate` | fallback when hashing is not possible or memory is short; sorts by key first |

**Notes:** `count_distinct` adds an extra aggregation and shuffle to de-duplicate; several distinct counts in one query
are planned with an `Expand`, multiplying rows. `approx_count_distinct` avoids both.

In [ ]:
print("\n".join(l for l in plan_lines(orders.groupBy("customer_id").agg(F.sum("amount"), F.collect_set("status"))) if "Aggregate" in l or "Exchange" in l))
print("\n".join(l.strip()[:80] for l in plan_lines(orders.agg(F.count_distinct("customer_id"), F.count_distinct("status"))) if "Expand" in l or "Aggregate" in l))

## Aggregate functions

Reduce many rows to one value, in `df.agg(...)`, in `groupBy(...).agg(...)` or over a window. All of them skip
nulls except `count("*")`. The examples group `orders` by customer or summarize the whole table.

| function | what it does |
| --- | --- |
| `count`, `count_distinct` | Rows (`count("*")`), non-null values (`count(col)`), and distinct values |
| `approx_count_distinct` | An estimate of the distinct count using HyperLogLog, with a target relative error (`rsd`, default 0.05) |
| `sum`, `avg` / `mean`, `min`, `max` | The standard summaries |
| `count_if` | Count rows where a condition is true |
| `sum_distinct` | The sum of the distinct values |
| `product` | All values multiplied together |
| `try_sum`, `try_avg` | `sum` and `avg` that return null on overflow instead of failing |
| `collect_list` / `array_agg`, `collect_set` | Gather a group's values into an array: all values, or distinct values |
| `listagg` / `string_agg` | Join a group's values into one string with a delimiter |
| `first`, `last`, `any_value` | A value from the group: the first or last one Spark reads, or any one |
| `max_by`, `min_by` | The value of one column on the row where another column is largest or smallest |
| `mode` | The most frequent value |
| `median`, `percentile` | Exact median and exact percentiles (pass a list for several), interpolated between values |
| `percentile_approx` / `approx_percentile` | An approximate percentile that is always a value from the data; higher `accuracy` costs more memory |
| `histogram_numeric` | An approximate histogram with `n` bins, as an array of `{x, y}` structs (bin centre, count) |
| `stddev`, `variance` and their population versions | Spread: `stddev` / `std` / `stddev_samp` and `variance` / `var_samp` treat rows as a sample; `stddev_pop` and `var_pop` as the whole population |
| `skewness`, `kurtosis` | Shape: skewness measures asymmetry (positive means a long right tail); kurtosis measures how extreme the tails are |
| `corr`, `covar_samp`, `covar_pop` | Pearson correlation (-1 to 1) and covariance between two columns |
| `regr_slope`, `regr_intercept`, `regr_r2`, `regr_count`, `regr_avgx`, `regr_avgy` | A least-squares line `y = slope * x + intercept` per group, its fit (r2) and the pairs it used |
| `bool_and` / `every`, `bool_or` / `some` | Whether a condition holds for every row, or for at least one row, of the group |
| `bit_or`, `bit_and`, `bit_xor` | Combine integer flags bit by bit across a group |
| `grouping`, `grouping_id` | In `rollup`, `cube` and grouping sets, mark which rows are subtotals |

### `count`, `count_distinct`

Rows (`count("*")`), non-null values (`count(col)`), and distinct values.

**Signature:** `F.count("*")` · `F.count(col)` · `F.count_distinct(*cols)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or `"*"` | `"*"` counts rows; a column counts its non-null values |
| `*cols` | Column or name | `count_distinct`: one or more columns (distinct combinations) |

**Use it for:** Volumes, completeness and cardinality.

**Notes:** `count(col)` and `count_distinct` skip nulls.

In [ ]:
customers.agg(F.count("*").alias("rows"), F.count("region").alias("with_region"), F.count_distinct("region").alias("regions")).show()

### `approx_count_distinct`

An estimate of the distinct count using HyperLogLog, with a target relative error (`rsd`, default 0.05).

**Signature:** `F.approx_count_distinct(col, rsd=0.05)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | column to count |
| `rsd` | float, default 0.05 | target relative standard deviation; smaller is more accurate and uses more memory |

**Use it for:** Distinct users per day on very large data.

**Notes:** Estimates cannot be added together: daily estimates do not sum to a monthly one.

In [ ]:
spark.range(50_000).select((F.col("id") % 4999).alias("user")).agg(F.count_distinct("user").alias("exact"),
    F.approx_count_distinct("user").alias("approx"), F.approx_count_distinct("user", 0.01).alias("approx_rsd_1pct")).show()

### `sum`, `avg` / `mean`, `min`, `max`

The standard summaries. `mean` is an alias of `avg`.

**Signature:** `F.sum(col)` · `F.avg(col)` · `F.mean(col)` · `F.min(col)` · `F.max(col)`

**Use it for:** Every metric table.

**Notes:** `sum` of no values is null, not 0; wrap it in `coalesce` for reports.

In [ ]:
orders.groupBy("customer_id").agg(F.sum("amount").alias("sum"), F.round(F.avg("amount"), 2).alias("avg"),
    F.min("amount").alias("min"), F.max("amount").alias("max")).orderBy("customer_id").show(3)

### `count_if`

Count rows where a condition is true.

**Signature:** `F.count_if(condition)`

**Use it for:** Several filtered counts in one pass.

**Notes:** A null condition is not counted.

In [ ]:
orders.groupBy("customer_id").agg(F.count_if(F.col("status") == "paid").alias("paid"), F.count_if(F.col("amount") > 100).alias("over_100")).orderBy("customer_id").show(3)

### `sum_distinct`

The sum of the distinct values.

**Signature:** `F.sum_distinct(col)`

**Use it for:** Rare: only when the values themselves should be unique.

**Notes:** It removes repeated **values**, not repeated rows; to undo duplicate rows, deduplicate by key and `sum`.

In [ ]:
spark.createDataFrame([(1, 50), (1, 50), (2, 30), (3, 30)], "payment_id int, amount int").agg(
    F.sum("amount").alias("sum"), F.sum_distinct("amount").alias("sum_distinct")).show()

### `product`

All values multiplied together.

**Signature:** `F.product(col)`

**Use it for:** Compounding growth factors.

**Notes:** The result is a `double`.

In [ ]:
spark.createDataFrame([(1.10,), (0.95,), (1.20,)], "factor double").agg(F.round(F.product("factor"), 4).alias("product")).show()

### `try_sum`, `try_avg`

`sum` and `avg` that return null on overflow instead of failing.

**Signature:** `F.try_sum(col)` · `F.try_avg(col)`

**Use it for:** Very large integer sums where overflow is possible.

**Notes:** A null result hides how large the true value was; cast to `decimal` or `bigint` when you need the number.

In [ ]:
spark.createDataFrame([(9223372036854775807,), (1,)], "v long").agg(F.try_sum("v").alias("try_sum"), F.try_avg("v").alias("try_avg")).show()

### `collect_list` / `array_agg`, `collect_set`

Gather a group's values into an array: all values, or distinct values.

**Signature:** `F.collect_list(col)` · `F.array_agg(col)` · `F.collect_set(col)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | values to gather |

**Use it for:** Lists of statuses, ids or tags per entity.

**Notes:** Element order is not guaranteed; wrap in `array_sort`. Large groups build large arrays in one task.

In [ ]:
orders.groupBy("customer_id").agg(F.array_sort(F.collect_list("status")).alias("statuses"),
    F.array_sort(F.collect_set("status")).alias("distinct_statuses")).orderBy("customer_id").show(3, truncate=False)

### `listagg` / `string_agg`

Join a group's values into one string with a delimiter.

**Signature:** `F.listagg(col, delimiter)` · `F.string_agg(col, delimiter)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | values |
| `delimiter` | str or Column, optional | separator, empty string by default |

**Use it for:** Readable report columns.

**Notes:** Order is not guaranteed; in SQL use `listagg(col, ',') WITHIN GROUP (ORDER BY ...)`.

In [ ]:
orders.groupBy("customer_id").agg(F.listagg("status", F.lit(", ")).alias("listagg")).orderBy("customer_id").show(3, truncate=False)
spark.sql("SELECT customer_id, listagg(status, ', ') WITHIN GROUP (ORDER BY order_ts) AS ordered FROM {o} GROUP BY customer_id ORDER BY customer_id LIMIT 3", o=orders).show(truncate=False)

### `first`, `last`, `any_value`

A value from the group: the first or last one Spark reads, or any one.

**Signature:** `F.first(col, ignorenulls=False)` · `F.last(col, ignorenulls=False)` · `F.any_value(col)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | values |
| `ignorenulls` | bool, default False | skip nulls |

**Use it for:** Carrying an attribute that is the same on every row of the group, such as a name per id.

**Notes:** Which row they read is not defined; to pick a specific row use `max_by`, `min_by` or a window.

In [ ]:
orders.groupBy("customer_id").agg(F.first("status").alias("first"), F.last("status").alias("last"), F.any_value("customer_id").alias("any_value")).orderBy("customer_id").show(3)

### `max_by`, `min_by`

The value of one column on the row where another column is largest or smallest.

**Signature:** `F.max_by(value_col, order_col)` · `F.min_by(value_col, order_col)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | the value to return |
| `ord` | Column or name | the column whose maximum or minimum picks the row |

**Use it for:** The latest status, the biggest order, the first product bought, without a join or a window.

**Notes:** Ties pick an arbitrary row; pass a struct as `order_col` to add a tie-breaker.

In [ ]:
orders.groupBy("customer_id").agg(F.max_by("order_id", "amount").alias("biggest_order"), F.max_by("status", "order_ts").alias("latest_status"),
    F.min_by("order_id", "order_ts").alias("first_order")).orderBy("customer_id").show(3)

### `mode`

The most frequent value. `deterministic=True` breaks ties by the smallest value.

**Signature:** `F.mode(col, deterministic=False)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | values |
| `deterministic` | bool, default False | break ties by the smallest value |

**Use it for:** The usual category, store or device per customer.

**Notes:** Without `deterministic=True`, a tie can give a different answer on each run.

In [ ]:
orders.agg(F.mode("status", deterministic=True).alias("most_common_status")).show()

### `median`, `percentile`

Exact median and exact percentiles (pass a list for several), interpolated between values.

**Signature:** `F.median(col)` · `F.percentile(col, p_or_list)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | numeric values |
| `percentage` | float or list | `percentile`: value(s) between 0 and 1 |
| `frequency` | int, default 1 | `percentile`: weight per row |

**Use it for:** Typical values that a few extremes would distort.

**Notes:** Exact percentiles hold all of a group's values in memory; use `percentile_approx` on big groups.

In [ ]:
orders.agg(F.median("amount").alias("median"), F.percentile("amount", [0.25, 0.75]).alias("quartiles")).show(truncate=False)

### `percentile_approx` / `approx_percentile`

An approximate percentile that is always a value from the data; higher `accuracy` costs more memory.

**Signature:** `F.percentile_approx(col, p_or_list, accuracy=10000)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | numeric values |
| `percentage` | float or list | value(s) between 0 and 1 |
| `accuracy` | int, default 10000 | higher is more accurate; relative error is about 1/accuracy |

**Use it for:** P95 latency and other percentiles over billions of rows.

**Notes:** Results can differ slightly from the exact percentile.

In [ ]:
orders.agg(F.percentile_approx("amount", 0.5).alias("approx_median"), F.approx_percentile("amount", [0.9, 0.99]).alias("p90_p99")).show(truncate=False)

### `histogram_numeric`

An approximate histogram with `n` bins, as an array of `{x, y}` structs (bin centre, count).

**Signature:** `F.histogram_numeric(col, F.lit(n))`

**Use it for:** A quick look at a distribution without leaving Spark.

**Notes:** Bin centres are computed from the data, not fixed boundaries.

In [ ]:
orders.agg(F.histogram_numeric(F.col("amount").cast("double"), F.lit(3)).alias("histogram")).show(truncate=False)

### `stddev`, `variance` and their population versions

Spread: `stddev` / `std` / `stddev_samp` and `variance` / `var_samp` treat rows as a sample; `stddev_pop` and `var_pop` as the whole population.

**Signature:** `F.stddev(col)` · `F.stddev_pop(col)` · `F.variance(col)` · `F.var_pop(col)`

**Use it for:** Variability and outlier thresholds.

**Notes:** A single-row group has a null sample standard deviation.

In [ ]:
orders.agg(F.round(F.stddev("amount"), 1).alias("stddev"), F.round(F.stddev_pop("amount"), 1).alias("stddev_pop"),
           F.round(F.variance("amount"), 0).alias("variance"), F.round(F.var_pop("amount"), 0).alias("var_pop")).show()

### `skewness`, `kurtosis`

Shape: skewness measures asymmetry (positive means a long right tail); kurtosis measures how extreme the tails are.

**Signature:** `F.skewness(col)` · `F.kurtosis(col)`

**Use it for:** Deciding whether an average is a fair summary.

**Notes:** Both are sensitive to a single outlier, which is what they are meant to reveal.

In [ ]:
orders.agg(F.round(F.skewness("amount"), 2).alias("skewness"), F.round(F.kurtosis("amount"), 2).alias("kurtosis")).show()

### `corr`, `covar_samp`, `covar_pop`

Pearson correlation (-1 to 1) and covariance between two columns.

**Signature:** `F.corr(a, b)` · `F.covar_samp(a, b)` · `F.covar_pop(a, b)`

**Use it for:** Checking whether two metrics move together.

**Notes:** Correlation does not show cause, and it only measures straight-line relationships.

In [ ]:
ads.groupBy("region").agg(F.round(F.corr("spend", "sales"), 3).alias("corr"), F.round(F.covar_samp("spend", "sales"), 1).alias("covar_samp"),
                          F.round(F.covar_pop("spend", "sales"), 1).alias("covar_pop")).orderBy("region").show()

### `regr_slope`, `regr_intercept`, `regr_r2`, `regr_count`, `regr_avgx`, `regr_avgy`

A least-squares line `y = slope * x + intercept` per group, its fit (r2) and the pairs it used.

**Signature:** `F.regr_slope(y, x)` and the others (note: **y first**)

| parameter | type | description |
| --- | --- | --- |
| `y` | Column or name | dependent variable (first!) |
| `x` | Column or name | independent variable |

**Use it for:** Simple trend lines per region or product without leaving Spark.

**Notes:** Rows where either value is null are skipped by every `regr_` function.

In [ ]:
ads.groupBy("region").agg(F.round(F.regr_slope("sales", "spend"), 2).alias("slope"), F.round(F.regr_intercept("sales", "spend"), 1).alias("intercept"),
    F.round(F.regr_r2("sales", "spend"), 3).alias("r2"), F.regr_count("sales", "spend").alias("pairs"),
    F.regr_avgx("sales", "spend").alias("avg_x"), F.regr_avgy("sales", "spend").alias("avg_y")).orderBy("region").show()

### `bool_and` / `every`, `bool_or` / `some`

Whether a condition holds for every row, or for at least one row, of the group.

**Signature:** `F.bool_and(cond)` · `F.every(cond)` · `F.bool_or(cond)` · `F.some(cond)`

**Use it for:** "all orders settled?", "any return?" flags per customer.

**Notes:** An empty or all-null group gives null.

In [ ]:
orders.groupBy("customer_id").agg(F.bool_and(F.col("status").isin("paid", "shipped")).alias("all_settled"),
    F.bool_or(F.col("status") == "returned").alias("any_return")).orderBy("customer_id").show(3)

### `bit_or`, `bit_and`, `bit_xor`

Combine integer flags bit by bit across a group.

**Signature:** `F.bit_or(col)` · `F.bit_and(col)` · `F.bit_xor(col)`

**Use it for:** Permission masks and feature flags stored as integers.

**Notes:** They work on integer types only.

In [ ]:
grants = spark.createDataFrame([("ana", 1), ("ana", 2), ("ben", 1), ("ben", 5)], "user string, permission int")
grants.groupBy("user").agg(F.bit_or("permission").alias("bit_or"), F.bit_and("permission").alias("bit_and"), F.bit_xor("permission").alias("bit_xor")).orderBy("user").show()

### `grouping`, `grouping_id`

In `rollup`, `cube` and grouping sets, mark which rows are subtotals.

**Signature:** `F.grouping(col)` · `F.grouping_id(*cols)`

| parameter | type | description |
| --- | --- | --- |
| `col / *cols` | Column or name | grouping columns of the rollup or cube |

**Use it for:** Telling a subtotal row from a row whose key really is null.

**Notes:** Never detect subtotals by testing for null keys.

In [ ]:
orders.rollup("status").agg(F.count("*").alias("orders"), F.grouping("status").alias("is_total")).orderBy("is_total", "status").show()

## Conditional aggregates

### `sum(when(...))`, `count_if` and SQL `FILTER`

Put the condition **inside** the aggregate to compute several filtered metrics in one pass instead of one query per
metric:

| form | example |
| --- | --- |
| `F.sum(F.when(cond, col))` | paid revenue: `F.sum(F.when(F.col("status") == "paid", F.col("amount")))` |
| `F.count_if(cond)` | number of returned orders |
| `F.avg(F.when(cond, 1).otherwise(0))` | share of rows matching `cond` |
| SQL `agg(col) FILTER (WHERE cond)` | `sum(amount) FILTER (WHERE status = 'paid')` |

**Notes:** `when` without `otherwise` gives null for other rows, which `sum` skips (good) but which makes the sum null
for a group with no matches; wrap it in `coalesce(..., 0)` if the report must show 0.

In [ ]:
orders.groupBy("customer_id").agg(
    F.coalesce(F.sum(F.when(F.col("status") == "paid", F.col("amount"))), F.lit(0)).alias("paid_revenue"),
    F.count_if(F.col("status") == "returned").alias("returns"),
    F.round(F.avg(F.when(F.col("status") == "returned", 1).otherwise(0)), 2).alias("return_rate"),
    F.expr("sum(amount) FILTER (WHERE status IN ('paid', 'shipped'))").alias("settled_revenue"),
).orderBy("customer_id").show()

## Approximate and mergeable aggregates

### HyperLogLog sketches: `hll_sketch_agg`, `hll_union_agg`, `hll_sketch_estimate`

`approx_count_distinct` gives one estimate, which cannot be combined later: daily estimates do not add up to a monthly
one. A **sketch** is a small binary summary that can be stored and merged: build one per day with `hll_sketch_agg`,
merge any range of days with `hll_union_agg`, and read the estimate with `hll_sketch_estimate`.

| function | returns |
| --- | --- |
| `F.hll_sketch_agg(col, lgConfigK=12)` | a binary sketch of the distinct values |
| `F.hll_union_agg(sketch_col)` | the union of many sketches (an aggregate) |
| `F.hll_union(a, b)` | the union of two sketches (row-wise) |
| `F.hll_sketch_estimate(sketch)` | the distinct-count estimate |

**Use it for:** distinct users per day stored in a summary table, then distinct users for any week or month without
rescanning raw events.

In [ ]:
daily = events.groupBy(F.to_date("ts").alias("day")).agg(F.hll_sketch_agg("user_id").alias("users_sketch"))
daily.select("day", F.hll_sketch_estimate("users_sketch").alias("users")).orderBy("day").show()
print("users across all days:", daily.agg(F.hll_sketch_estimate(F.hll_union_agg("users_sketch"))).first()[0],
      "| sum of daily counts:", daily.select(F.hll_sketch_estimate("users_sketch").alias("u")).agg(F.sum("u")).first()[0])

## Pivot and unpivot

### `GroupedData.pivot(pivot_col, values=None)`

Turns the distinct values of `pivot_col` into columns: one row per group key, one column per value, filled by the
aggregate passed to `agg`.

| parameter | type | description |
| --- | --- | --- |
| `pivot_col` | str | column whose values become columns |
| `values` | list, optional | the values to use, in this order. Always pass it: without it Spark runs an extra job to find them, and a new value in the data silently adds a column |

**Returns:** `GroupedData` (call `agg`, `sum`, `count`, ... next).

**Notes:** combinations with no rows are null, not 0; pivoting a high-cardinality column creates thousands of columns.
With several aggregates, columns are named `value_alias`. `count("*")` is not allowed inside a pivot; count a column.

In [ ]:
STATUSES = ["placed", "paid", "shipped", "returned"]
wide = orders.groupBy("customer_id").pivot("status", STATUSES).agg(F.sum("amount"))
wide.orderBy("customer_id").show()
orders.groupBy("customer_id").pivot("status", ["paid", "returned"]).agg(F.count("order_id").alias("n"), F.sum("amount").alias("rev")).orderBy("customer_id").show(3)

### `DataFrame.unpivot(ids, values, variableColumnName, valueColumnName)` / `melt`

The reverse of a pivot: turns columns into rows. Each row of the input produces one row per column in `values`.

| parameter | type | description |
| --- | --- | --- |
| `ids` | str or list | columns kept as identifiers |
| `values` | list or None | columns to turn into rows (None: all non-id columns) |
| `variableColumnName` | str | name of the column holding the former column names |
| `valueColumnName` | str | name of the column holding the values |

**Notes:** all `values` columns must share a type (cast first). Nulls are kept; filter them out if they mean "no row".

In [ ]:
wide.unpivot("customer_id", STATUSES, "status", "revenue").where("revenue IS NOT NULL").orderBy("customer_id", "status").show(5)

## Subtotals: rollup, cube and grouping sets

### `rollup`, `cube` and `GROUPING SETS`

| method | levels produced for keys (a, b) |
| --- | --- |
| `df.rollup("a", "b")` | (a, b), (a), () — hierarchical subtotals and a grand total |
| `df.cube("a", "b")` | (a, b), (a), (b), () — every combination |
| SQL `GROUP BY GROUPING SETS ((a), (b), ())` | exactly the listed levels |

In subtotal rows, the rolled-up keys are null. `F.grouping(col)` (1 when `col` is aggregated away) and
`F.grouping_id(*cols)` (a bit mask of the levels) identify each row's level; they are the only reliable way to tell a
subtotal from a row whose key really is null.

**Notes:** `cube` over n columns produces 2^n levels; prefer grouping sets when you only need some.

In [ ]:
with_region = orders.join(customers.select("customer_id", "region"), "customer_id", "left")
(with_region.rollup("region", "status")
 .agg(F.sum("amount").alias("revenue"), F.grouping_id().alias("level"))
 .orderBy("level", F.asc_nulls_last("region"), "status").show(25))
spark.sql("""
    SELECT region, status, sum(amount) AS revenue, grouping_id() AS level
    FROM {t} GROUP BY GROUPING SETS ((region), (status), ())
    ORDER BY level, region, status
""", t=with_region).show()

## Aggregations in SQL

### `GROUP BY`, `GROUP BY ALL`, `HAVING` and `FILTER`

Everything above has a SQL form. Spark SQL also supports `GROUP BY ALL` (group by every non-aggregate expression in
the `SELECT`), positional `GROUP BY 1, 2`, `HAVING`, the `FILTER (WHERE ...)` clause and `WITH ROLLUP` / `WITH CUBE`.

```sql
SELECT customer_id, status, count(*) AS orders, sum(amount) FILTER (WHERE amount > 100) AS big_revenue
FROM orders
GROUP BY ALL
HAVING count(*) >= 1
```

In [ ]:
spark.sql("""
    SELECT customer_id, count(*) AS orders, sum(amount) FILTER (WHERE amount > 100) AS big_revenue
    FROM {o}
    GROUP BY ALL
    HAVING count(*) >= 2
    ORDER BY customer_id
""", o=orders).show()

**Next:** practise these entries in [PySpark practice 7: aggregations](pyspark-practice-07-aggregations.ipynb).